In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
adata_scmono.write("/home/jaehyunchoi//Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono.h5ad")

In [ ]:
adata_scmono

In [ ]:
adata_scmono.obs['10_cluster'].value_counts()

In [ ]:
adata_scmono.obs['sample'].value_counts()

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='SLC7A5_expression',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='E2F2',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='SLC7A5_expression',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='10_cluster',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='subclustering_10',
    size=10
)

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scmono.obs['10_cluster']
batch_labels = adata_scmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'TIMD4',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'CX3CR1',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
sc.pl.dotplot(adata_scmono, ['SLC7A5','PIK3CA','PIK3CB','PIK3CD','AKT1','MTOR','PTEN','RHEB','INPP4A','INPP4B'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd
# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['10_cluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters, key=int):  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    ipf_frac = ipf_in / (ipf_in + ipf_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("cluster")
print(df_result)
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster'].astype(int)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster',groups=['5'],reference = 'rest', method='wilcoxon')
rank_5_df = sc.get.rank_genes_groups_df(adata_scmono, group='5')
rank_5_df

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scmono_DEG.csv")

In [ ]:
rank_5_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scmono5_DEG.csv")

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='state')
sc.tl.rank_genes_groups(adata_scmono, groupby='state', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df_state = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes_state = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes_state,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scmono_State_DEG.csv")

In [ ]:
rank_df_state

In [ ]:
import os
import tempfile

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi//Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono.h5ad")
adata_scmono

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='10_cluster',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='subclustering_10',
    size=15
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='state',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    size=15
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color=['SLC7A5','SLC3A2'],
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    size=17
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color=['MTOR'],
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    size=17
)

In [ ]:
adata_scmono.obs['monoMAC'] = adata_scmono.obs['10_cluster'].replace({
    '0': 'mono1',
    '1': 'KC1',
    '2': 'cDC2',
    '3': 'mono2',
    '4': 'KC2',
    '5': 'mono3',
    '6': 'mono4',
    '7': 'KC3',
    '8': 'cDC1',
    '9': 'cDC2'
})

sc.pl.umap(adata_scmono, color=['monoMAC'], legend_loc="on data")

In [ ]:
sc.pl.umap(adata_scmono, color=['monoMAC'])

In [ ]:
sc.pl.umap(adata_scmono, color=['monoMAC'])

In [ ]:
sc.pl.dotplot(adata_scmono, ['SLC7A5','SLC7A8','SLC43A1','SLC43A2'], 
              groupby="monoMAC",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['LY6G6C','ITGAM','CCR2','CX3CR1','S100A8','CLEC4F','TIMD4','MARCO','VSIG4','ITGAX','XCR1','ZBTB46','BATF3','IRF4'], 
              groupby="monoMAC",
              standard_scale=None,
              figsize=(5, 3))

In [ ]:
genes = ['E2F2'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SLC7A5','SLC7A8','SLC43A1','SLC43A2','SLC6A14'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.6 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
Transporters = {
    'Glucogenic': ['SLC6A19', 'SLC1A5', 'SLC7A8', 'SLC1A4', 'SLC38A1', 'SLC38A2'],
    'Ketogenic': ['SLC7A5', 'SLC16A10', 'SLC43A1', 'SLC3A2','SLC7A6']
}

In [ ]:
genes = ['SLC7A5', 'SLC16A10', 'SLC43A1', 'SLC3A2','SLC7A6'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.6 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SLC6A19', 'SLC1A5', 'SLC7A8', 'SLC1A4', 'SLC38A1', 'SLC38A2'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.6 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SLC7A5','SLC7A8','SLC43A1','SLC43A2','SLC6A14'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.6 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SLC7A5','SLC7A8','SLC43A1','SLC43A2'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
adata_scmono

In [ ]:
genes = ['SLC7A5','SLC3A2','RRAGA','DEPDC5','NPRL2','NPRL3','RHEB','MTOR','RPTOR','SREBF1','MLXIPL','PPARG','FASN','ACACA','INSR', 'AKT1', 'TSC2'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(2, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='state',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SLC7A5','SLC3A2','RRAGA','DEPDC5','NPRL2','NPRL3'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['RHEB','MTOR','RPTOR'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['SREBF1','MLXIPL','PPARG','FASN','ACACA'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='monoMAC',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Leucing - GcN2
genes = ['EIF2AK4','ATF4'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['GCLC','GPX2','HMOX1'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['KDM6B','ACLY','SIRT1','ATF4','IRF4','TBX21','GCLC','NFE2L2','SLC7A11','BECN1','ATG5','FOXO3','EP300','IL6','TNF'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='10_cluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.tl.rank_genes_groups(
    adata_scmono,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata_scmono, group="NAFLD")
print(deg_df.head())

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]

top20_filtered = filtered.sort_values(by='scores', ascending=False).head(20)
print(top20_filtered)

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5","SLC3A2","MTOR"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
desired_order = ["mono1", "mono2", "mono3", "mono4", "KC1", "KC2","KC3","cDC1","cDC2" ]
adata_scmono.obs['monoMAC'] = pd.Categorical(adata_scmono.obs['monoMAC'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.matrixplot(
    adata_scmono,
    Transporters,
    groupby="monoMAC",
    standard_scale= 'var'
)

In [ ]:
genes = ['SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2',]

In [ ]:
sc.pl.matrixplot(
    adata_scmono,
    genes,
    groupby="monoMAC",
    standard_scale= 'var'
)

In [ ]:
glucogen = ['AKT3','SIK','CREB3','ADCY2','PPARGC1A','CPT1C','CPT1A','CPT1B','CREB1','ATF2','CREBBP','PPP4R3C','CREB3L4','SIK1','LDHAL6A','CALML6','CRTC2','EP300','AKT1','AKT2','FBP1','FOXO1','SIK2','PLCB1','SIRT1','G6PC1','GCG','GCGR','GCK','GNAQ','GNAS','GYS1','GYS2','ACACA','ACACB','PRMT1','ITPR1','ITPR2','ITPR3','LDHA','LDHB','LDHC','PGAM4','ATF4','PCK1','PCK2','PDE3B','PRKAG2','PDHA1','PDHA2','PDHB','CALML5','PFKFB1','PFKL','PFKM','PFKP','PGAM1','PGAM2','PHKA1','PHKA2','PHKB','PHKG1','PHKG2','PKM','PLCB2','PLCB3','PLCB4','PRKAG3','PPARA','PPP3CA','PPP4C','PPP3CB','PPP3CC','PPP3R1','PPP3R2','PRKAA1','PRKAA2','PRKAB1','PRKAB2','PRKACA','PRKACB','PPP4R3A','PRKACG','PRKAG1','PPP4R3B','G6PC2','PYGB','PYGL','PYGM','CREB3L2','SLC2A1','SLC2A2','CALM1','CALM2','CALM3','CALML3','CAMK2A','CAMK2B','CAMK2D','CAMK2G','CREB3L3','FBP2','CREB3L1','CALML4','LDHAL6B','G6PC3','CREB5']

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=glucogen , score_name="glucogen_score")
sc.pl.umap(adata_scmono, color='glucogen_score', cmap='bwr', title='glucogen_score')

# SLC7A5 correlation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 유전자 이름 정의
geneA = "SLC7A5"
geneB = "TREM2"

# 인덱스 찾기
idxA = np.where(adata_scmono.var_names == geneA)[0][0]
idxB = np.where(adata_scmono.var_names == geneB)[0][0]

# log1p-normalized expression 값 불러오기
x = adata_scmono.X[:, idxA].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxA]
y = adata_scmono.X[:, idxB].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxB]

# 상관계수 계산
corr = np.corrcoef(x, y)[0, 1]

# 산점도 그리기
plt.figure(figsize=(6, 6))
plt.scatter(x, y, s=8, alpha=0.4)
plt.title(f"{geneA} vs {geneB}\nPearson r = {corr:.3f}")
plt.xlabel(f"{geneA} expression (log1p-normalized)")
plt.ylabel(f"{geneB} expression (log1p-normalized)")
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

geneA = "SLC7A5"
geneB = "TREM2"
idxA = np.where(adata_scmono.var_names == geneA)[0][0]
idxB = np.where(adata_scmono.var_names == geneB)[0][0]

x = adata_scmono.X[:, idxA].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxA]
y = adata_scmono.X[:, idxB].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxB]


R, pval = pearsonr(x, y)
df = pd.DataFrame({
    geneA: x,
    geneB: y
})


sns.set(style="whitegrid")

g = sns.lmplot(
    data=df,
    x=geneA,
    y=geneB,
    height=5, aspect=1.2,
    scatter_kws={'s':25, 'alpha':0.6, 'color':'black'},
    line_kws={'color':'red', 'lw':2}
)

# ✅ 회귀선의 95% 신뢰구간은 `sns.lmplot`이 자동으로 그림 (shade=True 기본값)
plt.title(
    f"{geneA} vs {geneB} expression (log1p-normalized)\n"
    f"Pearson R = {R:.2f}, p = {pval:.2e}",
    fontsize=12
)

plt.xlabel(f"{geneA} expression")
plt.ylabel(f"{geneB} expression")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

geneA = "SLC7A5"
geneB = "GPNMB"
idxA = np.where(adata_scmono.var_names == geneA)[0][0]
idxB = np.where(adata_scmono.var_names == geneB)[0][0]

x = adata_scmono.X[:, idxA].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxA]
y = adata_scmono.X[:, idxB].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxB]


R, pval = pearsonr(x, y)
df = pd.DataFrame({
    geneA: x,
    geneB: y
})


sns.set(style="whitegrid")

g = sns.lmplot(
    data=df,
    x=geneA,
    y=geneB,
    height=5, aspect=1.2,
    scatter_kws={'s':25, 'alpha':0.6, 'color':'black'},
    line_kws={'color':'red', 'lw':2}
)

# ✅ 회귀선의 95% 신뢰구간은 `sns.lmplot`이 자동으로 그림 (shade=True 기본값)
plt.title(
    f"{geneA} vs {geneB} expression (log1p-normalized)\n"
    f"Pearson R = {R:.2f}, p = {pval:.2e}",
    fontsize=12
)

plt.xlabel(f"{geneA} expression")
plt.ylabel(f"{geneB} expression")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

geneA = "SLC7A5"
geneB = "SPP1"
idxA = np.where(adata_scmono.var_names == geneA)[0][0]
idxB = np.where(adata_scmono.var_names == geneB)[0][0]

x = adata_scmono.X[:, idxA].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxA]
y = adata_scmono.X[:, idxB].toarray().flatten() if hasattr(adata_scmono.X, "toarray") else adata_scmono.X[:, idxB]


R, pval = pearsonr(x, y)
df = pd.DataFrame({
    geneA: x,
    geneB: y
})


sns.set(style="whitegrid")

g = sns.lmplot(
    data=df,
    x=geneA,
    y=geneB,
    height=5, aspect=1.2,
    scatter_kws={'s':25, 'alpha':0.6, 'color':'black'},
    line_kws={'color':'red', 'lw':2}
)

# ✅ 회귀선의 95% 신뢰구간은 `sns.lmplot`이 자동으로 그림 (shade=True 기본값)
plt.title(
    f"{geneA} vs {geneB} expression (log1p-normalized)\n"
    f"Pearson R = {R:.2f}, p = {pval:.2e}",
    fontsize=12
)

plt.xlabel(f"{geneA} expression")
plt.ylabel(f"{geneB} expression")

plt.tight_layout()
plt.show()